# DENUE Ciudad de México profiling

This notebook profiles the complete INEGI DENUE file for Mexico City (all 16 alcaldías) before the spatial join. All columns are read as strings with the published Latin-1 encoding so identifiers keep their leading zeros.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.transform.denue import DENUE_FILE, activity_group, sector_code

denue = pd.read_csv(DENUE_FILE, encoding='latin-1', dtype=str)
print(f'Rows: {len(denue):,}; columns: {len(denue.columns)}')
display(denue.head())

Rows: 462,732; columns: 42


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,12299604,09017112519000021000000000U5,ACUARIO AQUAVIRTUAL,NaN,112519,Otra acuicultura,0 a 5 personas,CALLE,PROCURADURIA GENERAL DE JUSTICIA,CALLE,...,Venustiano Carranza ...,1204,001,5584307089,KUINO2106@GMAIL.COM,NaN,Fijo,19.41887981,-99.09012648,2026-04
1,11845788,09013112515000091000000000U5,AJOLOTARIO APATLACO,NaN,112519,Otra acuicultura,0 a 5 personas,OTRO(ESPECIFIQUE),NINGUNO,CALLE,...,Apatlaco ...,1281,800,5577367690,EDUARDOSANCHEVALDERRAMA@GMAIL.COM,NaN,Fijo,19.26452374,-99.0895052,2025-05
2,11845785,09013112515000071000000000U7,AJOLOTARIO CARRIZAL,NaN,112519,Otra acuicultura,0 a 5 personas,OTRO(ESPECIFIQUE),NINGUNO,CALLE,...,Isla de las Muñecas ...,1281,800,5526720456,NaN,NaN,Fijo,19.27568509,-99.10182441,2025-05
3,11845786,09013112515000082000000000U4,AJOLOTARIO CHINAMPA ONKALI,ONKALI,112519,Otra acuicultura,6 a 10 personas,OTRO(ESPECIFIQUE),CANAL,CALLE,...,Xochimilco ...,1099,034,5518275086,NaN,NaN,Fijo,19.27748645,-99.09797138,2025-05
4,11845789,09013112515000102000000000U0,AJOLOTARIO TLAZOCAMA TONANA,NaN,112519,Otra acuicultura,6 a 10 personas,OTRO(ESPECIFIQUE),CANAL,CALLEJON,...,Xochimilco ...,1099,015,NaN,ORTADAVID294@GMAIL.COM,NaN,Fijo,19.27485335,-99.10180151,2025-05


## Identifiers and coordinates

Duplicate `id` values would violate the fact-table grain. Coordinates are converted only for profiling here; the production transform performs the authoritative validation and reprojection.

In [2]:
coordinates = denue[['latitud', 'longitud']].apply(pd.to_numeric, errors='coerce')
coordinate_profile = pd.DataFrame({
    'null_or_invalid': coordinates.isna().sum(),
    'zero': coordinates.eq(0).sum(),
    'minimum': coordinates.min(),
    'maximum': coordinates.max(),
})
print(f"Duplicated id values: {denue['id'].duplicated().sum():,}")
display(coordinate_profile)

Duplicated id values: 0


,null_or_invalid,zero,minimum,maximum
latitud,0,0,19.114837,32.519695
longitud,0,0,-116.931733,-98.945228


## SCIAN sectors and analytical groups

SCIAN sectors 31–33 (manufacturing) and 48–49 (transportation, postal services and warehousing) are official combined sectors. Sector 46 is **Retail**. Sectors 51–56, 61, 62, 71, 72 and 81 are **Services**. All remaining sectors are **Other**; sector 93 is excluded from Services because it represents government activities. Sector names in the warehouse are stored in English.

In [3]:
denue['sector_code'] = denue['codigo_act'].map(sector_code)
denue['activity_group'] = denue['sector_code'].map(activity_group)
sector_table = (denue.groupby(['sector_code', 'activity_group'], dropna=False)
                .size().rename('establishments').reset_index())
display(sector_table)
display(denue['activity_group'].value_counts().rename_axis('activity_group').to_frame('establishments'))

,sector_code,activity_group,establishments
0,11,Other,48
1,21,Other,41
2,22,Other,677
3,23,Other,1655
4,31-33,Other,32316
5,43,Other,15525
6,46,Retail,212251
7,48-49,Other,3844
8,51,Services,1729
9,52,Services,3977


,establishments
activity_group,
Retail,212251
Services,192925
Other,57556


## Employment bands

`per_ocu` is an employment-size band, not an exact employee count. The transform preserves the published Spanish labels because `dw.dim_business_size` uses them as natural keys.

In [4]:
display(denue['per_ocu'].value_counts(dropna=False).rename_axis('per_ocu').to_frame('establishments'))

,establishments
per_ocu,
0 a 5 personas,390466
6 a 10 personas,32397
11 a 30 personas,24418
31 a 50 personas,5786
51 a 100 personas,4630
101 a 250 personas,2999
251 y más personas,2036


## Registration dates

`fecha_alta` records the establishment's DENUE registration/start month; it is not the publication date of the current DENUE snapshot. The warehouse converts valid `YYYY-MM` values to the first day of that month solely to connect them to `dim_date`.

In [5]:
alta = pd.to_datetime(denue['fecha_alta'], format='%Y-%m', errors='coerce')
print(f'Valid registration months: {alta.notna().sum():,}')
print(f'Missing/invalid registration months: {alta.isna().sum():,}')
print(f'Range: {alta.min():%Y-%m} to {alta.max():%Y-%m}')
display(denue['fecha_alta'].value_counts(dropna=False).sort_index().tail(15).to_frame('establishments'))

Valid registration months: 462,696
Missing/invalid registration months: 36
Range: 2010-07 to 2026-04


,establishments
fecha_alta,
2019-04,2086
2019-11,83066
2020-04,1345
2020-11,2114
2021-05,1263
2021-11,170
2022-05,27
2022-11,208
2023-11,4641


## Production decision

The ETL sends the complete Mexico City file through coordinate validation and point-in-polygon assignment. Geometry, not DENUE's reported `cve_mun`, `cve_loc` and `ageb` codes, decides which urban AGEB an establishment belongs to; establishments outside every urban AGEB are dropped and counted. The reported state, municipality, locality and AGEB codes are retained as `cvegeo_reported` to audit the join.